# 04 · Findings & field map (the one-page summary)

This notebook pulls the three explorations together into the table you'll design the schema and extraction prompts from.

In [1]:
import sys, json, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so we can import recall_explore from the repo root
import pandas as pd
from recall_explore import fetch, load_samples, find_ids, coverage, normalize_model
pd.set_option("display.max_colwidth", 80)

LIVE = False   # True = call the real API (needs internet). False = use the 20 saved samples.

In [2]:
cpsc, fda, nh = load_samples("cpsc"), load_samples("openfda"), load_samples("nhtsa")

def pct(n, d): return f"{n}/{d} ({round(100*n/d)}%)"

cpsc_model = sum(1 for r in cpsc if find_ids(r["Description"])["models"])
cpsc_struct = sum(1 for r in cpsc for p in r["Products"] if p.get("Model"))
cpsc_upc = sum(1 for r in cpsc if r["ProductUPCs"])
food = [r for r in fda if r["_endpoint"] == "food"]
fda_upc = sum(1 for r in fda if find_ids(r["product_description"])["upcs"])
food_upc = sum(1 for r in food if find_ids(r["product_description"])["upcs"])
fda_lot = sum(1 for r in fda if find_ids(r["code_info"])["lots"])
fda_codes = sum(1 for r in fda if r["code_info"].strip().lower() not in ("no codes", "n/a", ""))

pd.DataFrame({
  "CPSC (20)": ["Description prose", pct(cpsc_struct, 20) + " structured · " + pct(cpsc_model, 20) + " in prose",
                pct(cpsc_upc, 20) + " (only 2019 recalls)", "dates/lots rare, in prose", "XML/JSON, no key, no paging"],
  "openFDA (20)": ["product_description + code_info", "devices only (catalog #)",
                   pct(fda_upc, 20) + " · food " + pct(food_upc, len(food)), pct(fda_lot, 20) + " regex · " + pct(fda_codes, 20) + " have codes",
                   "JSON, optional key, limit/skip paging, weekly"],
  "NHTSA (20)": ["Make/Model/ModelYear fields", "100% structured (vehicle level)", "n/a", "n/a (VIN-level only on nhtsa.gov)",
                 "JSON, no key, daily + bulk zip"],
}, index=["Where identifiers live", "Model #", "UPC", "Lot / serial / date", "Access"])

,CPSC (20),openFDA (20),NHTSA (20)
Where identifiers live,Description prose,product_description + code_info,Make/Model/ModelYear fields
Model #,0/20 (0%) structured · 13/20 (65%) in prose,devices only (catalog #),100% structured (vehicle level)
UPC,2/20 (10%) (only 2019 recalls),9/20 (45%) · food 8/10 (80%),n/a
Lot / serial / date,"dates/lots rare, in prose",12/20 (60%) regex · 19/20 (95%) have codes,n/a (VIN-level only on nhtsa.gov)
Access,"XML/JSON, no key, no paging","JSON, optional key, limit/skip paging, weekly","JSON, no key, daily + bulk zip"


## Decisions this exploration triggers
| # | Decision | Why (evidence) |
|---|---|---|
| 1 | **Extract identifiers from prose with an LLM**, regex as pre-filter | CPSC `Model` field empty in 93/93 recalls checked; openFDA lots are multi-item lists |
| 2 | **Keep raw payloads** in `recalls.raw` and re-extract when prompts improve | Formats vary wildly; extraction will get better over time |
| 3 | **Three match tiers**: exact (model/UPC), structured (vehicle), semantic (brand+product+photo) | 4 of 20 CPSC samples (1 in 5) have no identifier at all |
| 4 | **openFDA = lookup, not alert** source; link to FDA.gov | FDA disclaimer: not for public alerts; status not updated |
| 5 | **VIN = decode → model-level recalls + link to official VIN check** | Model query over-matches trims (e.g. 4xe-only recalls) |
| 6 | **Car seats from NHTSA bulk file** | Not reachable via recallsByVehicle |
| 7 | Use **severity signals**: FDA `classification`, NHTSA `parkIt`/`parkOutSide`, CPSC hazard text | Prioritise alerts |

## Schema tweaks to make in week 2
- `recall_products.lot_ranges` → store a list of `{lot, exp_date}` **and** date-range objects `{from, to, kind:"best_by"}`.
- Add `recall_products.udi` / `ndc` (devices & drugs) and `recalls.severity` (normalised from the three sources).
- Add `recalls.event_id` (openFDA grouping) and `recall_vehicles` (one campaign → many make/model/year rows).

In [3]:
# The normaliser that will produce recall_products.model_norm — and its first unit-test cases
cases = {"TL1302Z": "TL1302Z", "7927-OZ1": "7927OZ1", "RWX-RBP02": "RWXRBP02", "xj-200 b": "XJ200B",
         "MMT-8062": "MMT8062", "PAPA03001AI": "PAPA03001AI"}
for raw, expected in cases.items():
    got = normalize_model(raw)
    print(f"{raw:<12} -> {got:<12} {'✓' if got == expected else '✗ expected ' + expected}")

TL1302Z      -> TL1302Z      ✓
7927-OZ1     -> 7927OZ1      ✓
RWX-RBP02    -> RWXRBP02     ✓
xj-200 b     -> XJ200B       ✓
MMT-8062     -> MMT8062      ✓
PAPA03001AI  -> PAPA03001AI  ✓
